<a href="https://colab.research.google.com/github/Viktoria-coder-prog/program-learning/blob/main/%D0%B7%D0%B0%D0%B4%D0%B0%D0%BD%D0%B8%D0%B5_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from glob import glob
import sqlite3

In [3]:
df = pd.read_csv('/content/задание_4_dataset.csv')

In [4]:
print(f"Размерность таблицы: {df.shape[0]} строк, {df.shape[1]} столбцов\n")

Размерность таблицы: 21525 строк, 11 столбцов



In [6]:
types_df = pd.DataFrame(
    {
        "Тип данных": df.dtypes,
        "Непустые значения": df.notna().sum(),
        "Уникальные значения": df.nunique(),
    }
)
print(types_df)

                 Тип данных  Непустые значения  Уникальные значения
children              int64              21525                    8
days_employed       float64              19351                19351
dob_years             int64              21525                   58
education            object              21525                   15
education_id          int64              21525                    5
family_status        object              21525                    5
family_status_id      int64              21525                    5
gender               object              21525                    3
income_type          object              21525                    8
debt                  int64              21525                    2
total_income        float64              19351                19351


In [7]:
# Пропуски
missing_explicit = df.isna().sum()
missing_percent = (df.isna().mean() * 100).round(2)

missing_df = pd.DataFrame(
    {"Количество NaN": missing_explicit, "Процент NaN (%)": missing_percent}
)
print(missing_df)

                  Количество NaN  Процент NaN (%)
children                       0              0.0
days_employed               2174             10.1
dob_years                      0              0.0
education                      0              0.0
education_id                   0              0.0
family_status                  0              0.0
family_status_id               0              0.0
gender                         0              0.0
income_type                    0              0.0
debt                           0              0.0
total_income                2174             10.1


In [8]:
# Дубликаты
dup_rows = df.duplicated().sum()
print(f"Полных дубликатов строк: {dup_rows}")

Полных дубликатов строк: 647


In [11]:
# приведение к нижнему регистру
df.columns = df.columns.str.lower()

# удаление дублей
df.drop_duplicates(keep='last', inplace=True)
df.duplicated().sum()

np.int64(0)

In [12]:
df.columns

Index(['children', 'days_employed', 'dob_years', 'education', 'education_id',
       'family_status', 'family_status_id', 'gender', 'income_type', 'debt',
       'total_income'],
      dtype='object')

In [13]:
# исправление аномалий
df["children"] = df["children"].replace({-1: 1, 20: 2})

median_age = df[df["dob_years"] > 0]["dob_years"].median()
df["dob_years"] = df["dob_years"].replace(0, median_age).astype(int)

df["gender"] = df["gender"].replace("XNA", df["gender"].mode()[0])

In [14]:
# корректировка стажа
df["days_employed"] = df["days_employed"].abs()
df.loc[df["days_employed"] > 300000, "days_employed"] = (
    df.loc[df["days_employed"] > 300000, "days_employed"] / 24
)

In [16]:
# заполнение пропусков по доходу/занятости
df["total_income"] = df["total_income"].fillna(
    df.groupby("income_type")["total_income"].transform("median")
)
df["days_employed"] = df["days_employed"].fillna(
    df.groupby("income_type")["days_employed"].transform("median")
)

In [17]:
# обработка аномальных значений
p1_income = df["total_income"].quantile(0.01)
p99_income = df["total_income"].quantile(0.99)
df["total_income"] = np.clip(df["total_income"], p1_income, p99_income)

p1_days = df["days_employed"].quantile(0.01)
p99_days = df["days_employed"].quantile(0.99)
df["days_employed"] = np.clip(df["days_employed"], p1_days, p99_days)

In [20]:
df.head(10)

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income
0,1,8437.673028,42,высшее,0,женат / замужем,0,F,сотрудник,0,253875.639453
1,1,4024.803754,36,среднее,1,женат / замужем,0,F,сотрудник,0,112080.014102
2,0,5623.422610,33,Среднее,1,женат / замужем,0,M,сотрудник,0,145885.952297
3,3,4124.747207,32,среднее,1,женат / замужем,0,M,сотрудник,0,267628.550329
4,0,14177.753002,53,среднее,1,гражданский брак,1,F,пенсионер,0,158616.077870
5,0,926.185831,27,высшее,0,гражданский брак,1,M,компаньон,0,255763.565419
6,0,2879.202052,43,высшее,0,женат / замужем,0,F,компаньон,0,240525.971920
7,0,152.779569,50,СРЕДНЕЕ,1,женат / замужем,0,M,сотрудник,0,135823.934197
8,2,6929.865299,35,ВЫСШЕЕ,0,гражданский брак,1,F,сотрудник,0,95856.832424
9,0,2188.756445,41,среднее,1,женат / замужем,0,M,сотрудник,0,144425.938277
